# tcrkit — quick tour

One example per capability. Every function takes one value or many (list, dict,
DataFrame), returns a DataFrame, and reports bad rows in an `error` column.
All options: `docs/api.md`.

In [1]:
import pandas as pd

import tcrkit

## 1. `stitch` — V/J + CDR3 → full-length chain

In [2]:
tcrkit.stitch('TRBV19', 'TRBJ2-7', 'CASSIRSSYEQYF', chain='beta', squeeze=True)   # squeeze: one Series

sequence_aa         MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNL...
error                                                               -
stitchr_warnings    No valid leader region allele determined yet f...
Name: 0, dtype: str

A table works too; a `chain` column mixes alpha and beta. Other column names:
`col_mapping={'v': 'v_call'}`. Mouse: `species='mouse'`.

In [3]:
tcrs = pd.DataFrame({'v':     ['TRBV19',        'TRAV12-2'],
                     'j':     ['TRBJ2-7',       'TRAJ23'],
                     'cdr3':  ['CASSIRSSYEQYF', 'CAVNTGGGNKLTF'],
                     'chain': ['beta',          'alpha']})
tcrkit.stitch(tcrs, progress_bar=False)

,v,j,cdr3,chain,sequence_aa,error,stitchr_warnings
0,TRBV19,TRBJ2-7,CASSIRSSYEQYF,beta,MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNL...,-,No valid leader region allele determined yet f...
1,TRAV12-2,TRAJ23,CAVNTGGGNKLTF,alpha,MKSLRVLLVILWLQLSWVWSQQKEVEQNSGPLSVPEGAIASLNCTY...,-,No valid leader region allele determined yet f...


## 2. `add_cdr3_junctions` — restore C104 / F118 before stitching

`stitch` repairs nothing. The J gene decides the last residue (F, W or C):

In [4]:
tcrkit.add_cdr3_junctions(cdr3=['ASSIRSSYEQY', 'AVMDSNYQLI'], j=['TRBJ2-7', 'TRAJ33'])

,cdr3,j,cdr3_fixed,changed,junction,purity,n,level,error
0,ASSIRSSYEQY,TRBJ2-7,CASSIRSSYEQYF,True,C/F,0.988983,291300,J,None
1,AVMDSNYQLI,TRAJ33,CAVMDSNYQLIW,True,C/W,1.000000,65517,J,None


## 3. `anarci` — protein → V/J genes, CDRs, IMGT numbering

**Pass `allow={'A', 'B'}` for alpha/beta chains**, or an alpha chain can come back as delta:

In [5]:
alpha = ('AQKITQTQPGMFVQEKEAVTLDCTYDTSDPSYGLFWYKQPSSGEMIFLIYQGSYDQQNATEGRYSLNFQKARKSANL'
         'VISASQLGDSAMYFCAMSQLNSGNTPLVFGKGTRLSVIA')

pd.concat({'no allow': tcrkit.anarci(alpha),
           'allow A, B': tcrkit.anarci(alpha, allow={'A', 'B'})})[['chain_type', 'v_gene', 'j_gene', 'CDR3_aa']]

,,chain_type,v_gene,j_gene,CDR3_aa
no allow,0,D,TRAV14/DV4*01,TRDJ4*01,AMSQLNSGNTPLV
"allow A, B",0,A,TRAV14/DV4*01,TRAJ29*01,AMSQLNSGNTPLV


`collapse='region'` gives the FR / CDR segments (`per_residue=True`: one row per residue):

`domains=True` marks the whole chain: V gene, N/D, J gene and the constant domain
(numbered IMGT C-DOMAIN + 1000):

In [6]:
full = tcrkit.stitch('TRBV19', 'TRBJ2-7', 'CASSIRSSYEQYF', chain='beta', squeeze=True)['sequence_aa']
tcrkit.anarci(full, allow={'A', 'B'}, collapse='gene_segment', domains=True)

,gene_segment,start,end,length,imgt_start,imgt_end,sequence
0,L,1,19,19,NaN,NaN,MSNQVLCCVVLCFLGANTV
0,V,20,114,95,1,108,DGGITQSPKYLFRKEGQNVTLSCEQNLNHDAMYWYRQDPGQGLRLI...
0,N/D,115,116,2,109,110,RS
0,J,117,131,15,113,127,SYEQYFGPGTRLTVT
0,C,132,260,129,1001.8,1125,EDLNKVFPPEVAVFEPSEAEISHTQKATLVCLATGFFPDHVELSWW...
0,-,261,308,48,NaN,NaN,DCGFTSVSYQQGVLSATILYEILLGKATLYAVLVSALVLMAMVKRKDF


`show=True` prints it along the sequence instead (CDRs `1`, `2`, `3`). Options:
`show={'width': 100, 'color': False}`.

In [7]:
tcrkit.anarci(full, allow={'A', 'B'}, domains=True, show=True)

0  308 residues
              1         11        21        31        41        51
sequence      MSNQVLCCVVLCFLGANTVDGGITQSPKYLFRKEGQNVTLSCEQNLNHDAMYWYRQDPGQ
gene_segment  LLLLLLLLLLLLLLLLLLLVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV
region                           ffffffffffffffffffffffffff11111ffffffffff

              61        71        81        91        101       111
sequence      GLRLIYYSQIVNDFQKGDIAEGYSVSREKKESFPLTVTSAQKNPTAFYLCASSIRSSYEQ
gene_segment  VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVnnJJJJ
region        fffffff222222fffffffffffffffffffffffffffffffffffff3333333333

              121       131       141       151       161       171
sequence      YFGPGTRLTVTEDLNKVFPPEVAVFEPSEAEISHTQKATLVCLATGFFPDHVELSWWVNG
gene_segment  JJJJJJJJJJJCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCCC
region        3ffffffffffAAAAAAAAAAAAAAAAAAAAAA.BBBBBBBBBBB........CCCCCCC

              181       191       201       211       221       231
sequence      KEVHSGVSTDPQPLKEQPALN

## 4. `normalize_tcr` / `normalize_mhc` — tidy gene and allele names

In [8]:
tcrkit.normalize_tcr(['TCRBV20S1', 'TRAV23/6', 'junk'], log_failures=False)

,symbol,standardized,species,error
0,TCRBV20S1,TRBV20-1,human,NaN
1,TRAV23/6,TRAV23/DV6,human,NaN
2,junk,NaN,NaN,not recognised by tidytcells


In [9]:
tcrkit.normalize_mhc(['A*0201', 'H2-Kb', 'DRB1*04:01', 'bad'], errors='coerce')

,allele_input,mhc_prefix,species,gene,allele,mhc_class,error
0,A*0201,HLA,human,A,A0201,I,NaN
1,H2-Kb,H2,mouse,K,H2-Kb,I,NaN
2,DRB1*04:01,HLA,human,DRB1,DRB1*0401,II,NaN
3,bad,NaN,NaN,NaN,NaN,NaN,ParseError: Could not parse 'bad'


## 5. `mhc_sequence` / `list_mhc` — allele → protein sequence

Names are normalized first. Default: the extracellular part (`region='full'`,
`'mature'`, ...). `source='decodertcr'` uses DecoderTCR's sequences instead.

In [10]:
tcrkit.mhc_sequence(['HLA-A*02:01', 'A*0201', 'DRB1*04:01'])

,allele_input,allele,sequence,length,gene,mhc_class,species,regions,signal_peptide,extracellular,transmembrane,cytoplasmic,error
0,HLA-A*02:01,A0201,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...,283,A,I,human,extracellular,None,1-283,None,None,None
1,A*0201,A0201,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...,283,A,I,human,extracellular,None,1-283,None,None,None
2,DRB1*04:01,DRB1*0401,GDTRPRFLEQVKHECHFFNGTERVRFLDRYFYHQEEYVRFDSDVGE...,198,DRB1,II,human,extracellular,None,1-198,None,None,None


In [11]:
tcrkit.list_mhc('genes').head()

,gene,mhc_class,species,n_alleles,alleles
0,Epfu-DRA,II,big brown bat,1,Epfu-DRA
1,Epfu-DRB,II,big brown bat,1,Epfu-DRB
2,B2M,I,black flying fox,1,B2M_flyingfox
3,Ptal-N,I,black flying fox,1,Ptal-N*01
4,Trvu-UB,I,brushtail possum,1,Trvu-UB*01


## 6. `annotate_mhc` — MHC chain → gene, IMGT numbering, groove helices

Trust `gene`; `allele` is the closest bundled one (`allele_identity` 1.0 = exact).

In [12]:
a2 = tcrkit.mhc_sequence('A*02:01', region='full', squeeze=True)['sequence']
tcrkit.annotate_mhc(a2)[['gene', 'allele', 'allele_identity', 'mhc_class', 'domains', 'helix']]

,gene,allele,allele_identity,mhc_class,domains,helix
0,A,A0201,1.0,I,"G-ALPHA1, G-ALPHA2","74-110, 164-200"


`collapse=` cuts it into segments. `domains=True` also numbers α3 (IMGT C-LIKE + 1000);
the leader counts back from IMGT 1:

In [13]:
tcrkit.annotate_mhc(a2, collapse='domain', domains=True)

,domain,start,end,length,imgt_start,imgt_end,sequence
0,-,1,24,24,-24,-1,MAVMAPRTLVLLLSGALALTQTWA
0,G-ALPHA1,25,114,90,1,90,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...
0,G-ALPHA2,115,206,92,1,92,GSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAA...
0,C-LIKE,207,298,92,1001,1121,DAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDT...
0,-,299,365,67,NaN,NaN,EPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSY...


In [14]:
tcrkit.annotate_mhc(a2, collapse='topology')

,topology,start,end,length,imgt_start,imgt_end,sequence
0,signal_peptide,1,24,24,-24,-1,MAVMAPRTLVLLLSGALALTQTWA
0,extracellular,25,307,283,1,92,GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRME...
0,transmembrane,308,332,25,NaN,NaN,IVGIIAGLVLFGAVITGAVVAAVMW
0,cytoplasmic,333,365,33,NaN,NaN,RRKSSDRKGGSYSQAASSDSAQGSDVSLTACKV


`show=True` prints it. Use the `helix` line (docking-angle residues) as the groove
helices; region `H` is IMGT's wider 50–92:

In [15]:
tcrkit.annotate_mhc(a2, show=True)

0  365 residues
          1         11        21        31        41        51
sequence  MAVMAPRTLVLLLSGALALTQTWAGSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRF
topology  SSSSSSSSSSSSSSSSSSSSSSSSOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
domain                            111111111111111111111111111111111111
region                            AAAAAAAAAAAAAA...BBBBBBBBBBB..CCCCCC
helix

          61        71        81        91        101       111
sequence  DSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQ
topology  OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
domain    111111111111111111111111111111111111111111111111111111222222
region    CC...DDDDDDDDHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHAAAAAA
helix                  HHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHHH

          121       131       141       151       161       171
sequence  RMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQL
topology  OOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOOO
domain    2222

`imgt_regions()` lists the IMGT ranges behind CDRs and helices, with sources:

In [16]:
tcrkit.imgt_regions()

,scheme,mhc_class,chain,domain,region,column,imgt_start,imgt_end,skipped,source
0,tcr,-,-,V-DOMAIN,CDR1,region,27,38,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
1,tcr,-,-,V-DOMAIN,CDR2,region,56,65,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
2,tcr,-,-,V-DOMAIN,CDR3,region,105,117,,IMGT V-DOMAIN FR/CDR definition: https://www.i...
3,mhc,I,heavy,G-ALPHA1,helix,helix,50,86,,tcrkit helix='imgt': the groove-helix residues...
4,mhc,I,heavy,G-ALPHA2,helix,helix,52,86,,tcrkit helix='imgt': the groove-helix residues...
5,mhc,II,alpha,G-ALPHA,helix,helix,50,86,,tcrkit helix='imgt': the groove-helix residues...
6,mhc,II,beta,G-BETA,helix,helix,52,86,"61B, 62",tcrkit helix='imgt': the groove-helix residues...


Don't know what a sequence is? `annotate` says (`chain_type`) and annotates it as its kind. As with `anarci`, pass `allow={'A', 'B'}` when you expect alpha/beta TCRs; without it this alpha chain is called `TCR_delta`:

In [17]:
tcrkit.annotate({'TCR alpha': alpha, 'HLA-A2': a2, 'peptide': 'GILGFVFTL'},
                allow={'A', 'B'}, progress_bar=False)[['chain_type', 'species', 'v_gene', 'gene', 'allele', 'domains']]

,chain_type,species,v_gene,gene,allele,domains
TCR alpha,TCR_alpha,human,TRAV14/DV4*01,NaN,NaN,V
HLA-A2,MHC_I,human,NaN,A,A0201,"G-ALPHA1, G-ALPHA2, C-LIKE"
peptide,peptide,NaN,NaN,NaN,NaN,NaN


`annotate_mhc` takes the `allele` from the [MHC Motif Atlas](http://mhcmotifatlas.org) (downloaded once; `tcrkit download` gets it ahead of time): the gene first, then only that gene's alleles. `allele_ties` counts alleles just as close; class I is compared over the groove:

In [18]:
tcrkit.annotate_mhc(a2)[['gene', 'allele', 'allele_identity', 'allele_ties', 'allele_source']]

,gene,allele,allele_identity,allele_ties,allele_source
0,A,A0201,1.0,121,motifatlas


Its sequences are a source too. Class I alleles there are only the groove, so ask for `region='groove'`:

In [19]:
tcrkit.mhc_sequence(['B*51:250', 'DRB1*04:25'], source='motifatlas', region='groove')[['allele', 'length', 'regions', 'error']]

,allele,length,regions,error
0,B51250,181,extracellular,None
1,DRB1*0425,90,extracellular,None


And its ligands say which peptide residues probably anchor it: class I from the allele's ligands of that length, class II from the binding cores (the peptide's core is found first). `plot_motif` draws the logo, with `pip install 'tcrkit[logos]'`:

In [20]:
tcrkit.peptide_anchors(['GILGFVFTL', 'SIINFEKL', 'PKYVKQNTLKLAT'], ['HLA-A*02:01', 'H2-Kb', 'DRB1*01:01'])[
    ['peptide', 'allele', 'n_ligands', 'core', 'anchors', 'anchor_residues']]

,peptide,allele,n_ligands,core,anchors,anchor_residues
0,GILGFVFTL,HLA-A*02:01,14363,NaN,"P2, P9","I, L"
1,SIINFEKL,H2-Kb,4686,NaN,"P5, P8","F, L"
2,PKYVKQNTLKLAT,DRB1*01:01,29074,YVKQNTLKL,"P1, P4, P6, P9","Y, Q, T, L"


## 7. `build_tcr_pmhc` — all steps over a table

normalize → junctions → stitch → MHC → ANARCI check. Row `b` (bad V gene, serotype
`A2`) is reported; row `a` is still built:

In [21]:
raw = pd.DataFrame([
    {'id': 'a', 'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33', 'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'TCRBV20S1', 'beta_j': 'TRBJ2-7', 'beta_cdr3': 'ASSLGQAYEQY', 'mhc_a': 'HLA-A*02:01'},
    {'id': 'b', 'alpha_v': 'TRAV1-2', 'alpha_j': 'TRAJ33', 'alpha_cdr3': 'AVMDSNYQLI',
     'beta_v': 'NOTAGENE', 'beta_j': 'TRBJ2-7', 'beta_cdr3': 'ASSLGQAYEQY', 'mhc_a': 'A2'},
])
built = tcrkit.build_tcr_pmhc(raw, progress_bar=False)
built[['id', 'beta_v', 'mhc_a_allele', 'alpha_cdr3_ok', 'beta_cdr3_ok', 'ok', 'error']]

,id,beta_v,mhc_a_allele,alpha_cdr3_ok,beta_cdr3_ok,ok,error
0,a,TRBV20-1,A0201,True,True,True,-
1,b,NOTAGENE,A2,True,NaN,False,beta V call not recognised | beta did not stit...


Next: `tcrkit_metrics.ipynb` (repertoire metrics), `docs/api.md`, `tcrkit --help`.